### functon 1 讀取檔案
input ： path, is_display(bool)


output： 物件

In [1]:
import os
from OCC.Display.SimpleGui import init_display
from OCC.Extend.DataExchange import read_iges_file

def read_file(path, is_display=False):
    
    if os.path.exists(path):           # 防止路徑不存在
        shapes = read_iges_file(path)  # 一定要用這個方法讀取
    else:
        print("檔案路徑錯誤！")
    
    if is_display:
        # 初始化 3D 顯示環境
        display, start_display, add_menu, add_function_to_menu = init_display()
        display.DisplayShape(shapes, update=True)
        start_display()
        
    return shapes

In [2]:
path = r"C:\NTHU\IRTI-Project\simple_2.IGS"#物件路徑
object_1 = read_file(path, is_display=0)

### functon 2 分割物件平面
input ： object, save_path, is_save(bool)

output： 面 1 ~ 面 n

In [3]:
import os
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopoDS import topods
from OCC.Core.TopAbs import TopAbs_FACE
from OCC.Display.OCCViewer import OffscreenRenderer
from OCC.Core.GProp import GProp_GProps
from OCC.Core.BRepGProp import brepgprop_SurfaceProperties
from OCC.Core.BRepAlgoAPI import BRepAlgoAPI_Fuse
from OCC.Core.TopoDS import topods_Face, TopoDS_Compound, TopoDS_Shape

# 平面影像儲存路徑
image_directory = r"C:\NTHU\IRTI-Project\Images"
# display, start_display, add_menu, add_function_to_menu = init_display()
def fuse_faces(faces_to_fuse):

    result_shape = faces_to_fuse[0]  # 從列表中的第一個面開始
    for face in faces_to_fuse[1:]:  # 遍歷剩下的面
        fuse = BRepAlgoAPI_Fuse(result_shape, face)
        fuse.Build()
        if fuse.IsDone():
            result_shape = fuse.Shape()
    return result_shape


def plane_segmentation(object, save_path, is_save=True, area_threshold=100):
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    explorer = TopExp_Explorer(object, TopAbs_FACE)
    faces = []
    merge_indices=[3, 4, 6]    
    
    while explorer.More():
        face = topods.Face(explorer.Current())
        faces.append(face)
        explorer.Next()

    # 合併指定索引的面
    if merge_indices:
        faces_to_fuse = [faces[i] for i in merge_indices]
        fused_face = fuse_faces(faces_to_fuse)
        
        # 更新面列表：移除原始面，添加合併後的面
        for index in sorted(merge_indices, reverse=True):
            del faces[index]  # 從列表中移除
        faces.append(fused_face)  # 添加合併後的面

    # 儲存面和相關的影像
    index = 1
    renderer = OffscreenRenderer()
    renderer.Create()
    for face in faces:
        props = GProp_GProps()
        brepgprop_SurfaceProperties(face, props)
        area = props.Mass()
        if area > area_threshold:
            renderer.EraseAll()
            # if index == 5:
            #     display.DisplayShape(face, update=True, color='BLUE', transparency=0.7)
            #     start_display()
            renderer.DisplayShape(face, update=True, color='BLUE', transparency=0.5)
            renderer.FitAll()
            if is_save:
                image_path = os.path.join(save_path, f'object_face_{index}.png')
                renderer.View.Dump(image_path)
                print(f'Saved image of face {index} to {image_path}')
            index += 1
    
    
    return faces

In [4]:
object_1_faces = plane_segmentation(object_1, image_directory)
print("object_1 總共有", len(object_1_faces), "個物件")
# print(object_1_faces[0])
# print(object_1_faces[-1])

C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\2137224504.py:54: DeprecationWarning: Call to deprecated function brepgprop_SurfaceProperties since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method brepgprop.SurfaceProperties
  brepgprop_SurfaceProperties(face, props)
C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\2137224504.py:54: DeprecationWarning: Call to deprecated function brepgprop_SurfaceProperties since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method brepgprop.SurfaceProperties
  brepgprop_SurfaceProperties(face, props)


Many colors for color name BLUE, using first.
Saved image of face 1 to C:\NTHU\IRTI-Project\Images\object_face_1.png
Many colors for color name BLUE, using first.
Saved image of face 2 to C:\NTHU\IRTI-Project\Images\object_face_2.png
Many colors for color name BLUE, using first.
Saved image of face 3 to C:\NTHU\IRTI-Project\Images\object_face_3.png
Many colors for color name BLUE, using first.
Saved image of face 4 to C:\NTHU\IRTI-Project\Images\object_face_4.png
Many colors for color name BLUE, using first.
Saved image of face 5 to C:\NTHU\IRTI-Project\Images\object_face_5.png
object_1 總共有 5 個物件


C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\2137224504.py:54: DeprecationWarning: Call to deprecated function brepgprop_SurfaceProperties since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method brepgprop.SurfaceProperties
  brepgprop_SurfaceProperties(face, props)
C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\2137224504.py:54: DeprecationWarning: Call to deprecated function brepgprop_SurfaceProperties since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method brepgprop.SurfaceProperties
  brepgprop_SurfaceProperties(face, props)
C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\2137224504.py:54: DeprecationWarning: Call to deprecated function brepgprop_SurfaceProperties since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method brepgprop.SurfaceProperties
  brepgprop_SurfaceProperties(face, props)


### functon 3 平面+線段 生成 drive plane
input ： 平面 選擇的邊線 

output： 面 1 ~ 面 n

In [5]:
## 計算線段曲率
from OCC.Core.BRepLProp import BRepLProp_CLProps
from OCC.Core.BRepAdaptor import BRepAdaptor_Curve
from OCC.Core.GeomAPI import GeomAPI_ProjectPointOnSurf
from OCC.Core.GCPnts import GCPnts_UniformAbscissa
from OCC.Core.GCPnts import GCPnts_AbscissaPoint

from OCC.Core.GeomLProp import GeomLProp_SLProps
from OCC.Core.gp import gp_Pnt, gp_Vec

def calculate_average_curvature(edge, curve_length, props, point_spacing, num_points=10):
    # 將交線轉換成曲線
    curve_adaptor = BRepAdaptor_Curve(edge)
    first_param = curve_adaptor.FirstParameter() 
    last_param = curve_adaptor.LastParameter()   
    num_points = int(curve_length / point_spacing) + 1
    curvature_sum = 0
    
    for i in range(1, num_points - 1):  # 第一個跟最後一個點不算，怕有問題
        param = first_param + i * (last_param - first_param) * point_spacing / curve_length 
        point = curve_adaptor.Value(param)                                                 # 依序得到座標(從UV轉為XYZ)
        """
        它從你設計的這個面（比如一個彎曲的盾牌表面）中，取出它的數學模型，讓我們可以計算和操作它，例如測量它的彎曲程度或找到它的中心點等。
        """                                                 
        surface_handle = BRep_Tool.Surface(face)                                           
        projector = GeomAPI_ProjectPointOnSurf(point, surface_handle) ##在尋找如果從這個點垂直投影到曲面上，它會落在哪里
        
        if projector.NbPoints() > 0:  ##確保至少有一個投影點
            """
            在曲面上，每個點都可以通過兩個參數（通常表示為 u 和 v）來定位。這些參數有點像地圖上的經緯度，它們告訴你這個點在曲面的「網格」上的位置。
            """
            u, v = projector.LowerDistanceParameters()
        props.SetParameters(u, v)     ##告訴分析器在曲面上的哪個點進行計算
        if props.IsCurvatureDefined():
            curvature = props.MaxCurvature()
            
            
        curvature_sum += curvature    
        
    if props.IsCurvatureDefined():
            max_dir, min_dir = gp_Dir(), gp_Dir()
            props.CurvatureDirections(max_dir, min_dir)
            # 最大最小曲率的方向?
            print("Direction of Maximum Curvature:", max_dir.X(), max_dir.Y(), max_dir.Z())
            print("Direction of Minimum Curvature:", min_dir.X(), min_dir.Y(), min_dir.Z())

    average_curvature = curvature_sum / (num_points - 2)  ## 計算平均曲率
    # print(face_index + "\t" + path_index + "\t" + average_curvature)  
    return average_curvature
    

# 根據取率調整採樣點的距離

In [6]:
from OCC.Core.GeomLProp import GeomLProp_SLProps
from OCC.Core.gp import gp_Pnt, gp_Vec
import math
def calculate_suitable_plane_spacing(edge, curve_length, props, point_spacing, face, num_points=10, x=0.5):
    # 將交線轉換成曲線
    curve_adaptor = BRepAdaptor_Curve(edge)
    first_param = curve_adaptor.FirstParameter() 
    last_param = curve_adaptor.LastParameter()   
    num_points = int(curve_length / point_spacing) + 1
    curvature_list = []
    
    for i in range(1, num_points - 1):  # 第一個跟最後一個點不算，怕有問題
        param = first_param + i * (last_param - first_param) * point_spacing / curve_length 
        point = curve_adaptor.Value(param)                                                 # 依序得到座標(從UV轉為XYZ)
        """
        它從你設計的這個面（比如一個彎曲的盾牌表面）中，取出它的數學模型，讓我們可以計算和操作它，例如測量它的彎曲程度或找到它的中心點等。
        """                                                 
        surface_handle = BRep_Tool.Surface(face)                                           
        projector = GeomAPI_ProjectPointOnSurf(point, surface_handle) ##在尋找如果從這個點垂直投影到曲面上，它會落在哪里
        
        if projector.NbPoints() > 0:  ##確保至少有一個投影點
            """
            在曲面上，每個點都可以通過兩個參數（通常表示為 u 和 v）來定位。這些參數有點像地圖上的經緯度，它們告訴你這個點在曲面的「網格」上的位置。
            """
            u, v = projector.LowerDistanceParameters()
        props.SetParameters(u, v)     ##告訴分析器在曲面上的哪個點進行計算
        if props.IsCurvatureDefined():
            max_curvature = props.MaxCurvature()
            min_curvature = props.MinCurvature()
            
            # 取得最大曲率和最小曲率的絕對值
            abs_max_curvature = abs(max_curvature)
            abs_min_curvature = abs(min_curvature)

            # 比較絕對值大小，並將較大的曲率絕對值添加到列表中
            if abs_max_curvature < abs_min_curvature:
                curvature_list.append(abs_min_curvature)
            else:
                curvature_list.append(abs_max_curvature)
               
    # print("face_index=", face_index)
    # print("curvature_list=", curvature_list)
    max_curvature = max(curvature_list) ## 得到曲線上的最大曲率
    # print("max_curvature=", max_curvature)
    # min_curvature = min(curvature_list) ## 得到曲線上的最大曲率
    # print("min_curvature=", min_curvature)
    if max_curvature == 0:              ## 反推得到曲率半徑
        min_radius = 1000           
    else:
        min_radius = 1. / max_curvature
    # print("min_radius=", min_radius)
    min_spacing = math.sin(math.acos(1-(x/min_radius))) * 2 * min_radius  ## 得到最大容許長度
    return min_spacing

# 根據最小曲率方向生成目標線段

In [7]:
import os
import numpy as np
from OCC.Core.IGESControl import IGESControl_Reader
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopoDS import topods_Face, TopoDS_Face, TopoDS_Shape
from OCC.Core.BRepAdaptor import BRepAdaptor_Surface
from OCC.Core.BRepLProp import BRepLProp_SLProps
from OCC.Core.TopoDS import topods_Edge
from OCC.Core.TopAbs import TopAbs_FACE
from OCC.Core.TopAbs import TopAbs_EDGE
from OCC.Core.gp import gp_Pnt, gp_Vec
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeEdge, BRepBuilderAPI_MakeVertex
from OCC.Core.Bnd import Bnd_Box
from OCC.Core.BRepBndLib import brepbndlib_Add
from OCC.Core.gp import gp_Pnt
from OCC.Core.BRepPrimAPI import BRepPrimAPI_MakeBox
from OCC.Core.BRepExtrema import BRepExtrema_DistShapeShape
from OCC.Display.OCCViewer import rgb_color
from OCC.Core.gp import gp_Pnt, gp_Dir, gp_Lin
from OCC.Core.BRepAlgoAPI import BRepAlgoAPI_Section
from OCC.Core.TopAbs import TopAbs_VERTEX
from OCC.Core.BRep import BRep_Tool
def make_line(start_point, end_point):
    edge_builder = BRepBuilderAPI_MakeEdge(start_point, end_point)
    return edge_builder.Edge()


def extend_to_boundary(face, direction, step=0.1, max_steps=10000, tolerance=0.01):
    bbox = Bnd_Box()
    brepbndlib_Add(face, bbox)
    xmin, ymin, zmin, xmax, ymax, zmax = bbox.Get()
    
    center = gp_Pnt((xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2)
    point = center
    adjustment = 1
    xmin_adjusted = xmin - adjustment if xmax - xmin > 10 and xmin > 0 else xmin + adjustment if xmax - xmin > 10 and xmin < 0 else xmin
    ymin_adjusted = ymin - adjustment if ymax - ymin > 10 and ymin > 0 else ymin + adjustment if ymax - ymin > 10 and ymin < 0 else ymin
    zmin_adjusted = zmin - adjustment if zmax - zmin > 10 and zmin > 0 else zmin + adjustment if zmax - zmin > 10 and zmin < 0 else zmin
    xmax_adjusted = xmax - adjustment if xmax - xmin > 10 and xmax > 0 else xmax + adjustment if xmax - xmin > 10 and xmax < 0 else xmax
    ymax_adjusted = ymax - adjustment if ymax - ymin > 10 and ymax > 0 else ymax + adjustment if ymax - ymin > 10 and ymax < 0 else ymax
    zmax_adjusted = zmax - adjustment if zmax - zmin > 10 and zmax > 0 else zmax + adjustment if zmax - zmin > 10 and zmax < 0 else zmax
    box = BRepPrimAPI_MakeBox(gp_Pnt(xmin_adjusted, ymin_adjusted, zmin_adjusted),gp_Pnt(xmax_adjusted, ymax_adjusted, zmax_adjusted)).Shape()
    for _ in range(max_steps):
        next_point = gp_Pnt(point.X() + direction.X() * step,
                            point.Y() + direction.Y() * step,
                            point.Z() + direction.Z() * step)
        vertex = BRepBuilderAPI_MakeVertex(next_point).Vertex()
        dist = BRepExtrema_DistShapeShape(box, vertex)
        if dist.Value() > tolerance:
            break
        point = next_point
    # display.DisplayShape(face)
    
    # display.DisplayShape(box,update=True,color=rgb_color(0,0,0.1),transparency=1)
    return point

def process_face(face, use_min_tangent):
    adaptor_surface = BRepAdaptor_Surface(face, True)
    props = BRepLProp_SLProps(adaptor_surface, 2, 0.01)
    u_min, u_max, v_min, v_max = adaptor_surface.FirstUParameter(), adaptor_surface.LastUParameter(), adaptor_surface.FirstVParameter(), adaptor_surface.LastVParameter()

    u_values = np.linspace(u_min, u_max, num=100)
    v_values = np.linspace(v_min, v_max, num=100)

    max_CurvatureDirection_set = []
    min_CurvatureDirection_set = []
    max_curvature_set = []

    for u in u_values:
        for v in v_values:
            point = adaptor_surface.Value(u, v)
            vertex = BRepBuilderAPI_MakeVertex(point).Vertex()
            dist_shape_shape = BRepExtrema_DistShapeShape(vertex, face)
            dist_shape_shape.Perform()
            min_distance = dist_shape_shape.Value()

            if min_distance < 0.01:
                props.SetParameters(u, v)
                if props.IsCurvatureDefined():
                    max_curvature = props.MaxCurvature()
                    min_curvature = props.MinCurvature()
                    # 取得最大曲率和最小曲率的絕對值
                    abs_max_curvature = abs(max_curvature)
                    abs_min_curvature = abs(min_curvature)

                    # 比較絕對值大小，並將較大的曲率絕對值添加到列表中
                    if abs_max_curvature < abs_min_curvature:
                        max_curvature_set.append(abs_min_curvature)
                    else:
                        max_curvature_set.append(abs_max_curvature)
                        
                    max_dir, min_dir = gp_Dir(), gp_Dir()
                    props.CurvatureDirections(max_dir, min_dir)

                    max_CurvatureDirection = gp_Vec(max_dir.X(), max_dir.Y(), max_dir.Z())
                    min_CurvatureDirection = gp_Vec(min_dir.X(), min_dir.Y(), min_dir.Z())

                    max_CurvatureDirection_set.append(max_CurvatureDirection)
                    min_CurvatureDirection_set.append(min_CurvatureDirection)

    if use_min_tangent:
        average_tangent = gp_Vec(sum(tangent.X() for tangent in min_CurvatureDirection_set) / len(min_CurvatureDirection_set),
                            sum(tangent.Y() for tangent in min_CurvatureDirection_set) / len(min_CurvatureDirection_set),
                            sum(tangent.Z() for tangent in min_CurvatureDirection_set) / len(min_CurvatureDirection_set))
        average_tangent = average_tangent.Normalized()
    else:
        average_tangent = gp_Vec(sum(tangent.X() for tangent in max_CurvatureDirection_set) / len(max_CurvatureDirection_set),
                            sum(tangent.Y() for tangent in max_CurvatureDirection_set) / len(max_CurvatureDirection_set),
                            sum(tangent.Z() for tangent in max_CurvatureDirection_set) / len(max_CurvatureDirection_set))
        average_tangent = average_tangent.Normalized()
        
    return average_tangent, np.mean(max_curvature_set)

def generate_line_segments(shape, use_min_tangent=True):
    if isinstance(shape, TopoDS_Compound):
        weighted_vectors = gp_Vec(0, 0, 0)
        total_max_curvature = 0
        explorer = TopExp_Explorer(shape, TopAbs_FACE)
        while explorer.More():
            
            face = topods_Face(explorer.Current())
            average_tangent, average_maxcurvature = process_face(face, use_min_tangent)
            if average_tangent:
                weighted_vectors += average_tangent * average_maxcurvature
                total_max_curvature += average_maxcurvature
            explorer.Next()
        if total_max_curvature > 0:
            final_tangent = (weighted_vectors / total_max_curvature).Normalized()
            print("Final weighted average tangent = ", final_tangent.X(), final_tangent.Y(), final_tangent.Z())        
        else:
            final_tangent = average_tangent.Normalized()
            print("Final weighted average tangent = ", final_tangent.X(), final_tangent.Y(), final_tangent.Z())
        end_point_selected = extend_to_boundary(face, final_tangent)
        start_point_selected = extend_to_boundary(face, gp_Vec(-final_tangent.X(), -final_tangent.Y(), -final_tangent.Z()))
        line_selected = make_line(start_point_selected, end_point_selected)

        return line_selected 
        
        
    else:
        face = topods_Face(shape)
        average_tangent, _ = process_face(face, use_min_tangent)
        print("average_tangent = ",average_tangent.X(), average_tangent.Y(), average_tangent.Z())
        end_point_selected = extend_to_boundary(face, average_tangent)
        start_point_selected = extend_to_boundary(face, gp_Vec(-average_tangent.X(), -average_tangent.Y(), -average_tangent.Z()))
        line_selected = make_line(start_point_selected, end_point_selected)

        return line_selected   

In [8]:
#確定法向量方向
from OCC.Core.gp import gp_Pnt
from OCC.Core.GC import GC_MakeSegment
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeEdge, BRepBuilderAPI_MakeWire
from OCC.Core.gp import gp_Pnt, gp_Vec, gp_Dir,gp_Circ, gp_Ax2, gp_Pnt
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeEdge, BRepBuilderAPI_MakeWire,BRepBuilderAPI_MakeFace
from OCC.Core.BRepOffsetAPI import BRepOffsetAPI_MakePipe
from OCC.Display.SimpleGui import init_display
import numpy as np
from OCC.Core.IGESControl import IGESControl_Reader
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeEdge
from OCC.Core.BRepAlgoAPI import BRepAlgoAPI_Section
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_VERTEX
from OCC.Core.BRep import BRep_Tool
from OCC.Extend.DataExchange import read_iges_file


def check_normal_vector_direction(face,model):

    adaptor_surface = BRepAdaptor_Surface(face, True)
    props = BRepLProp_SLProps(adaptor_surface, 2, 0.01)
    u_min, u_max, v_min, v_max = adaptor_surface.FirstUParameter(), adaptor_surface.LastUParameter(), adaptor_surface.FirstVParameter(), adaptor_surface.LastVParameter()
    u_mid = (u_min + u_max) / 2
    v_mid = (v_min + v_max) / 2
    
    props.SetParameters(u_mid, v_mid)
    center = adaptor_surface.Value(u_mid, v_mid)
    if props.IsCurvatureDefined():
        normal = props.Normal()    
        print("normal = ", normal.X(), normal.Y(), normal.Z())
    # if projector.NbPoints() > 0:
    #     u, v = projector.LowerDistanceParameters()
    
    # else:
    #     print("No projection found for this point")        
    #     props.SetParameters(u, v)
    #     if props.IsCurvatureDefined():
    #         normal = props.Normal()
    # 創建一條線段
    point_s=gp_Pnt(center.X(),center.Y(), center.Z())
    point_e=gp_Pnt(float(center.X()+10000*normal.X()),float(center.Y()+10000*normal.Y()), float(center.Z()+10000*normal.Z()))
    line = BRepBuilderAPI_MakeEdge(point_s, point_e).Edge()

    # 計算交點
    section = BRepAlgoAPI_Section(line, model)
    section.Build()

    # 提取交點
    intersection_points = []
    if section.IsDone():
        ex = TopExp_Explorer(section.Shape(), TopAbs_VERTEX)
        while ex.More():
            vertex = ex.Current()
            point = BRep_Tool.Pnt(vertex)
            
            dist_checker = BRepExtrema_DistShapeShape(vertex, face)
            dist_checker.Perform()
            
            if dist_checker.IsDone() and dist_checker.NbSolution() > 0:
                min_distance = dist_checker.Value()
                if min_distance < 0.01:
                    print(f"Point ({point.X()}, {point.Y()}, {point.Z()}) is too close to the surface with distance {min_distance}. Ignoring.")
                else:
                    intersection_points.append(point)
                    print(f"Intersection point: ({point.X()}, {point.Y()}, {point.Z()})")
            
            ex.Next()

    # 打印交點數量和座標
    print(f"Number of intersection points: {len(intersection_points)}")

    
    # print(f"Number of intersection points: {len(intersection_points)}")
    for point in intersection_points:
        print(f"Intersection point: ({point.X()}, {point.Y()}, {point.Z()})")
        
    if len(intersection_points)%2==0:
        return 0
    else :
        return 1

In [9]:
from OCC.Core.gp import gp_Pnt, gp_Dir, gp_Pln, gp_Vec
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeFace
from OCC.Core.BRepAdaptor import BRepAdaptor_Curve
from OCC.Core.GCPnts import GCPnts_UniformAbscissa
from OCC.Display.SimpleGui import init_display
from OCC.Core.BRepAlgoAPI import BRepAlgoAPI_Section
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopoDS import topods_Face
from OCC.Core.GCPnts import GCPnts_AbscissaPoint
from OCC.Core.GCPnts import GCPnts_UniformAbscissa  
from OCC.Core.BRepLProp import BRepLProp_SLProps
from OCC.Core.BRepAdaptor import BRepAdaptor_Surface
from OCC.Core.GeomAPI import GeomAPI_ProjectPointOnSurf
from OCC.Core.BRep import BRep_Tool
from OCC.Core.TopAbs import TopAbs_FACE, TopAbs_REVERSED

def calculate_section(object, face, length, curve_adaptor, name = '', plane_spacing = 10.0, point_spacing = 5.0, is_save=True, is_display=False):
    """
    可以更容易地獲取這個面的各種幾何信息，比如曲面的尺寸、形狀、位置等，並且可以更方便地執行一些複雜的操作，如計算曲率等。
    """
    adaptor_surface = BRepAdaptor_Surface(face, True) 
    props = BRepLProp_SLProps(adaptor_surface, 2, 0.01)  # 這個分析器被配置為能夠計算高達二階導數的幾何性質（這由第二個參數指定，此處為 2），這意味著它能夠計算曲率等性質
    num_planes = int(length / plane_spacing) + 1
    path_index = 0
    check_normal=check_normal_vector_direction(face,object)

    if length % plane_spacing != 0:
        iter_plane = num_planes + 1
    else:
        iter_plane = num_planes
    
    for i in range(0, iter_plane):
        path_index += 1
        u = curve_adaptor.FirstParameter() + i * plane_spacing
        if u >= curve_adaptor.LastParameter():
            u = curve_adaptor.LastParameter()
        pnt = curve_adaptor.Value(u)           ## 得到XYZ座標
        tangent_vector = curve_adaptor.DN(u, 1).Normalized() ##  tangent_vector。DN(u, 1) 表示取得曲線在 u 點的第一階導數（即切向量），並且通過 Normalized() 函數將此向量標準化
        tangent_dir = gp_Dir(tangent_vector)  #表示為一個方向，不具備長度
        
        perpendicular_plane = gp_Pln(pnt, tangent_dir)   ##定義一個幾何平面，用點跟法向量，無限大的平面
        
        # if is_display:
        #     perpendicular_face = BRepBuilderAPI_MakeFace(perpendicular_plane, -110, 110, -110, 110).Face()
        #     display.DisplayShape(perpendicular_face, update=True, color='RED', transparency=0.9)
            
        section = BRepAlgoAPI_Section(face, perpendicular_plane, False) #用於計算兩個B-Rep幾何實體之間交集的物件
        """
        具體來說，當你對兩個幾何形體（例如，兩個面）進行交集操作來查找它們的交界線時，ComputePCurveOn1(True) 指示算法在第一個面上計算這條交界線的投影。
        這意味著它將生成一個參數化曲線，描述交界線如何在第一個面的局部坐標系中行進
        """
        section.ComputePCurveOn1(True)
        section.Build()
        
        if section.IsDone() :
            intersection_shape = section.Shape()
            if is_display:
                display.DisplayShape(intersection_shape, update=True, color='GREEN')
                
            explorer = TopExp_Explorer(intersection_shape, TopAbs_EDGE)
            if not explorer.More():
                print("No edges found in the intersection shape.",face_index, path_index)
                path_index -= 1
                continue
            edge = topods_Edge(explorer.Current())
            curve = BRepAdaptor_Curve(edge)    ## BRepAdaptor_Curve 是一個適配器類，用於從拓撲邊提取底層的曲線幾何資訊
                
            curve_length = GCPnts_AbscissaPoint.Length(curve, curve.FirstParameter(), curve.LastParameter())
            if curve_length <= 1:
                print("No edges found in the intersection shape.",face_index, path_index)
                path_index -= 1
                continue
            print(str(face_index) + '\t' + str(path_index) + '\t' + "curve_length:", curve_length)


            suitable_spacing = calculate_suitable_plane_spacing(edge, curve_length, props =props, face=face, point_spacing=1, num_points=1000) 
            print(str(face_index) + '\t' + str(path_index) + '\t' + "spacing:", suitable_spacing)
            if suitable_spacing < point_spacing:   ## 如果設定長度小於最大容許長度，則更改設定長度
                adjust_point_spacing = round(suitable_spacing-1)
            else:
                adjust_point_spacing = point_spacing
                
            # average_curvature = calculate_average_curvature(edge, curve_length, props =props, point_spacing=1.0, num_points=1000)
            # if average_curvature is not None:
            #     print(str(face_index) + '\t' + str(path_index) + '\t' + "Average curvature:", average_curvature)
            # else:
            #     print("No valid points found to calculate curvature.")
                
            num_points = int(curve_length / adjust_point_spacing) + 1
            
            if curve_length % adjust_point_spacing != 0:
                iter_point = num_points + 1
            else:
                iter_point = num_points
    
            for j in range(0, iter_point):
                param = curve.FirstParameter() + j * (curve.LastParameter() - curve.FirstParameter()) * adjust_point_spacing / curve_length #因為有些UV被正規畫到0~1之間了
                if param >= curve.LastParameter():
                    param = curve.LastParameter()
                point = curve.Value(param)
                if is_display:
                    display.DisplayShape(point, update=True, color='YELLOW')
                
            
                surface_handle = BRep_Tool.Surface(face)
                projector = GeomAPI_ProjectPointOnSurf(point, surface_handle)
                
                if projector.NbPoints() > 0:
                    u, v = projector.LowerDistanceParameters()
                else:
                    print("No projection found for this point")
                    
                props.SetParameters(u, v)
                if props.IsCurvatureDefined():
                    normal = props.Normal()
                
                #確定法向量皆朝外
                
                if check_normal == 1:#加一個判斷式
                    # print(normal.X(), normal.Y(), normal.Z())
                    # print("reversed")
                    normal.Reverse()
                    # print(normal.X(), normal.Y(), normal.Z())
                    
                    
                if is_save:
                    with open(name  + '.txt', 'a') as outfile:
                        outfile.write(str(face_index) + '\t' + str(path_index) +
                                    '\t' + format(point.X(), ".2f") + '\t' + format(point.Y(), ".2f") + '\t' + format(point.Z(), ".2f") +
                                    '\t' + format(normal.X(), ".2f") + '\t' + format(normal.Y(), ".2f") + '\t' + format(normal.Z(), ".2f") + '\n')
        
    
                    with open(name + '.csv', 'a', newline='') as outfile:
                        # 寫入數據，使用逗號分隔
                        outfile.write(f"{face_index},{path_index}," +
                                    f"{format(point.X(), '.2f')},{format(point.Y(), '.2f')},{format(point.Z(), '.2f')}," +
                                    f"{format(normal.X(), '.2f')},{format(normal.Y(), '.2f')},{format(normal.Z(), '.2f')}\n")
        
    

In [10]:
def path_planning(object, face, length, curve_adaptor, name = '', plane_spacing = 10.0, point_spacing = 5.0, is_save=True, is_display=True):
    if isinstance(face, TopoDS_Compound):
        explorer = TopExp_Explorer(face, TopAbs_FACE)
        while explorer.More():
            split_face = topods_Face(explorer.Current())
            calculate_section(object=object,face=split_face, length=length, curve_adaptor=curve_adaptor, name=name, plane_spacing=plane_spacing, point_spacing=point_spacing
                    , is_save=is_save, is_display=is_display)
            explorer.Next()
    else:
        calculate_section(object=object,face=face, length=length, curve_adaptor=curve_adaptor, name=name, plane_spacing=plane_spacing, point_spacing=point_spacing
                    , is_save=is_save, is_display=is_display)
            

In [11]:
import pandas as pd

def data_prcessing(data_name=""):
    data = pd.read_csv('simple_2.txt', sep='\t', header=None)
    data.columns = ['FaceIndex', 'PathIndex', 'X', 'Y', 'Z', 'NX', 'NY', 'NZ']

    data_sorted = data.sort_values(by=['FaceIndex', 'PathIndex'])

    merged_paths = pd.DataFrame()
    for face, group in data_sorted.groupby('FaceIndex'):
        for path, sub_group in group.groupby('PathIndex'):
            merged_paths = pd.concat([merged_paths, sub_group])

    merged_paths.to_csv('merged_paths.txt', sep='\t', index=False, header=False)

In [12]:
face_index = 0
display, start_display, add_menu, add_function_to_menu = init_display()
display.DisplayShape(object_1, update=True)
for face in object_1_faces:
    # face = topods_Face(face)
    face_index += 1
    
    selected_edge = generate_line_segments(face, use_min_tangent=1) ################
    
    curve_adaptor = BRepAdaptor_Curve(selected_edge)
    length = curve_adaptor.LastParameter() - curve_adaptor.FirstParameter()
    
    path_planning(object_1,face=face, length=length, curve_adaptor=curve_adaptor, name="simple_2", plane_spacing = 10.0, point_spacing =10.0
                    , is_save=True, is_display=True)
    
    data_prcessing(data_name="simple_2")
start_display()

C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\1467039719.py:141: DeprecationWarning: Call to deprecated function topods_Face since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method topods.Face
  face = topods_Face(shape)


average_tangent =  0.0 1.0 0.0
normal =  1.0 0.0 0.0
Point (-50.0, 0.0, 0.0) is too close to the surface with distance 0.0. Ignoring.
Intersection point: (50.0, 0.0, 0.0)
Number of intersection points: 1
Intersection point: (50.0, 0.0, 0.0)


C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\1467039719.py:30: DeprecationWarning: Call to deprecated function brepbndlib_Add since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method brepbndlib.Add
  brepbndlib_Add(face, bbox)
C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\1467039719.py:30: DeprecationWarning: Call to deprecated function brepbndlib_Add since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method brepbndlib.Add
  brepbndlib_Add(face, bbox)
C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\4127034557.py:65: DeprecationWarning: Call to deprecated function topods_Edge since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method topods.Edge
  edge = topods_Edge(explorer.Current())


1	1	curve_length: 100.0
1	1	spacing: 63.23764701504603


C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\4127034557.py:65: DeprecationWarning: Call to deprecated function topods_Edge since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method topods.Edge
  edge = topods_Edge(explorer.Current())


...

5	9	curve_length: 78.53980827544966
5	9	spacing: 14.106735979665885


C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\4127034557.py:65: DeprecationWarning: Call to deprecated function topods_Edge since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method topods.Edge
  edge = topods_Edge(explorer.Current())


5	10	curve_length: 78.53980827544966
5	10	spacing: 14.106735979665885


C:\Users\nick2\AppData\Local\Temp\ipykernel_28464\4127034557.py:65: DeprecationWarning: Call to deprecated function topods_Edge since pythonocc-core 7.7.1. This function will be removed in a future release, please rather use the static method topods.Edge
  edge = topods_Edge(explorer.Current())


5	11	curve_length: 78.53980827544966
5	11	spacing: 14.106735979665885
